# 🎃 Festa em Cena
## Análise exploratória e qualidade dos dados

**Mini-Projeto • Módulo 1 • Resolução comentada**

Vamos investigar 1.500 pedidos fictícios de fantasias e acessórios, usando **Pandas, expressões regulares e datas**.

> **Pergunta central:** os dados estão consistentes e como os pedidos se distribuem entre canais e períodos?

### Nosso percurso
1. Ler e conhecer os dados.
2. Limpar categorias e padronizar canais.
3. Validar quantidades e preços.
4. Converter datas com tratamento de erros.
5. Testar a hipótese sobre entregas.
6. Produzir contagens e interpretar os resultados.

**Como usar:** deixe `halloween_vendas.csv` na mesma pasta deste notebook. No VS Code, selecione um kernel Python com Pandas instalado e use **Executar Tudo**. Também é possível usar o Jupyter. As saídas já estão incluídas para consulta.

## 📘 Como ler este código

**Função** é um bloco que executa uma tarefa. Algumas vêm prontas, como `len()`; outras são criadas por nós com `def`.

**Método** é uma função associada a um objeto: em `texto.strip()`, pedimos ao texto que execute `strip`. Em `pd.read_csv()`, usamos uma função disponível no módulo Pandas.

**Atributo** guarda uma característica e normalmente é acessado sem parênteses: `dados.shape`, por exemplo. **Operador** combina ou compara valores, como `>`, `&`, `|` e `~`.

> Em cada etapa, veja primeiro **qual problema queremos resolver**. Depois observe qual função ou método nos ajuda. Não é necessário decorar todos os nomes.


In [1]:
# Execute esta célula somente se precisar instalar o Pandas no kernel atual.
# Descomente a linha abaixo removendo o primeiro #.
# %pip install pandas

### Ferramentas que vamos usar

| Recurso | O que faz | Por que usamos aqui |
|---|---|---|
| `Path()` e `.is_file()` | Representam um caminho e verificam se é um arquivo | Localizar o CSV e orientar quando ele não está na pasta |
| `datetime` | Representa datas e horários | Interpretar e apresentar datas |
| `Decimal` | Representa números decimais | Validar preços sem conversão para ponto flutuante binário |
| `re` | Trabalha com expressões regulares | Limpar pontuação e validar inteiros |
| `pd.set_option()` | Ajusta a exibição do Pandas | Facilitar a leitura das tabelas; não altera valores |
| `display()` e `Markdown()` | Exibem tabelas e texto formatado | Organizar as saídas do notebook |
| `raise FileNotFoundError(...)` | Interrompe com uma mensagem clara | Explicar que falta o arquivo necessário |

`import` carrega uma biblioteca; `from ... import ...` seleciona recursos dela. `pd` é o apelido usado para Pandas.


In [2]:
from pathlib import Path
from datetime import datetime
from decimal import Decimal, InvalidOperation
import re
import pandas as pd
from IPython.display import display, Markdown

# Opções de apresentação: não alteram os dados.
pd.set_option('display.max_columns', 20)
pd.set_option('display.max_colwidth', 65)

ARQUIVO = Path('halloween_vendas.csv')
if not ARQUIVO.is_file():
    raise FileNotFoundError('Coloque halloween_vendas.csv na mesma pasta do notebook.')

## 01 · Conhecer antes de transformar

Cada linha representa **um pedido de um único produto**, possivelmente com várias unidades. Portanto, contar linhas não é o mesmo que somar unidades.

| Campo | Significado |
|---|---|
| `pedido_id` | Identificador do pedido |
| `categoria` | Grupo do produto |
| `canal_venda` | Canal onde o pedido foi registrado |
| `quantidade` | Número de unidades |
| `preco_unitario` | Preço de uma unidade |
| `data_venda` | Data e hora local de São Paulo |

Vamos ler todas as colunas como texto. Com `keep_default_na=False`, os campos vazios ficam como `""`, permitindo verificar ausências explicitamente. Não sobrescreveremos o CSV original.

### Leitura: função, argumentos e retorno

| Recurso | O que faz | Por que usamos |
|---|---|---|
| `pd.read_csv()` | Lê o CSV e retorna um DataFrame | Transformar o arquivo em uma tabela para análise |
| `sep=','` | Define a vírgula como separador | Corresponder ao formato do arquivo |
| `encoding='utf-8'` | Define a codificação | Ler corretamente caracteres como `í` |
| `dtype=str` | Lê as colunas como texto | Validar antes de converter números e datas |
| `keep_default_na=False` | Não converte automaticamente marcadores em nulos | Manter vazios como strings para validação explícita |
| `.copy(deep=True)` | Cria uma cópia dos dados | Separar a tabela original da tabela tratada |
| `len(dados)` | Retorna o número de linhas | Conferir o tamanho da base |
| `.nunique()` | Conta valores distintos | Conferir quantos identificadores de pedido existem |
| `.head(3)` | Seleciona as primeiras três linhas | Inspecionar uma amostra |

`dados['pedido_id']` seleciona uma coluna, chamada **Series** no Pandas. `f'...'` cria um texto com valores de variáveis.


In [3]:
original = pd.read_csv(
    ARQUIVO, sep=',', encoding='utf-8', dtype=str, keep_default_na=False
)
dados = original.copy(deep=True)
print(f'Linhas: {len(dados):,} | Pedidos distintos: {dados["pedido_id"].nunique():,}')
display(dados.head(3))

Linhas: 1,500 | Pedidos distintos: 1,500


,pedido_id,data_venda,produto,categoria,canal_venda,cidade,quantidade,preco_unitario,custo_unitario,desconto_pct,status_pedido,publico,tamanho,data_entrega
0,844,2025-11-21 18:02:35,Fantasia de vampiro,Fantasia,Loja física,Curitiba,3,132.5,72.08,15,Cancelado,Adulto,G,
1,378,2024-10-29 18:41:26,Fantasia de bruxa,FANTASIA!!!,Loja física,Maringá,3,110,57,5,Concluído,Adulto,M,2024-11-03 18:41:26
2,953,2025-09-05 21:16:09,Fantasia de bruxa,Fantasia,Web,Maringá,2,116.6,60.42,5,Concluído,Adulto,M,2025-09-10 21:16:09


### Ausência não é sempre a mesma coisa

| Recurso | O que faz | Por que usamos |
|---|---|---|
| `.isna()` | Marca nulos do Pandas | Verificar ausências representadas como `NaN` ou `None` |
| `.str.strip()` | Remove espaços das extremidades de cada texto | Identificar também campos contendo apenas espaços |
| `.eq('')` | Compara cada valor com uma string vazia | Detectar os vazios mantidos pela leitura |
| `.sum()` sobre booleanos | Soma `True` como 1 e `False` como 0 | Contar registros que atendem à condição |
| `.apply(lambda coluna: ...)` | Executa uma função em cada coluna, neste uso | Repetir a mesma verificação nas colunas |
| `pd.DataFrame({...})` | Constrói uma tabela a partir de um dicionário | Comparar nulos e vazios lado a lado |

**Exemplo:** `'   '.strip()` resulta em `''`. Uma string vazia não é detectada por `.isna()`.


In [4]:
# isna() identifica nulos do Pandas; strings vazias precisam de outra verificação.
ausencias = pd.DataFrame({
    'nulos_pandas': original.isna().sum(),
    'vazios_ou_espacos': original.apply(lambda coluna: coluna.str.strip().eq('').sum())
})
display(ausencias)

,nulos_pandas,vazios_ou_espacos
pedido_id,0,0
data_venda,0,0
produto,0,0
categoria,0,18
canal_venda,0,0
cidade,0,0
quantidade,0,0
preco_unitario,0,12
custo_unitario,0,7
desconto_pct,0,0


## 02 · Categorias e canais consistentes

A categoria passa por `.strip()`, `.lower()` e `re.sub()`. A expressão `[^a-zA-Z\s]` encontra caracteres que **não são letras nem espaços**. As categorias desta base não usam acentos.

Categorias inicialmente vazias são contadas separadamente das preenchidas que mudaram. Uma mudança apenas de maiúsculas para minúsculas também conta como alteração.

### Função `limpar_categoria(texto)`

**Problema:** a mesma categoria pode aparecer com espaços, maiúsculas e pontuação.

**Recebe:** uma string. **Retorna:** a categoria padronizada, também como string.

| Etapa | O que faz | Por que usamos |
|---|---|---|
| `texto.strip()` | Remove espaços nas extremidades | Eliminar espaços que não pertencem ao nome |
| `.lower()` | Converte para minúsculas | Fazer `Fantasia` e `FANTASIA` coincidirem |
| `re.sub(padrao, '', texto)` | Substitui os caracteres encontrados por vazio | Remover pontuação, preservando letras e espaços |
| `limpa.strip() or 'sem categoria'` | Usa o texto se estiver preenchido; senão usa o rótulo | Tratar também textos que viram vazios após a limpeza |

**Exemplo:** `limpar_categoria('  FANTASIA!!!  ')` → `'fantasia'`.

`def` define a função; `return` devolve o resultado. Aqui a expressão regular é apropriada porque as categorias preenchidas não contêm acentos.


In [5]:
def limpar_categoria(texto):
    """Remove pontuação, preservando letras e espaços entre palavras."""
    limpa = re.sub(r'[^a-zA-Z\s]', '', texto.strip().lower())
    return limpa.strip() or 'sem categoria'


### Função `quantidade_valida(texto)`

**Problema:** nem todo texto representa uma quantidade inteira positiva.

**Recebe:** o texto da quantidade. **Retorna:** `True` ou `False`.

`re.fullmatch(r'[+-]?\d+', texto)` exige que **todo o texto** corresponda ao padrão: sinal opcional (`[+-]?`) seguido de um ou mais dígitos (`\d+`). `bool()` transforma a correspondência em verdadeiro ou falso. `int(texto) > 0` exige quantidade positiva.

**Por que `and` aqui?** Trabalhamos com um texto por vez. O segundo teste só executa se o primeiro passar, evitando tentar converter um texto inválido.

**Exemplos:** `'3'` → `True`; `'-1'`, `'2.5'` e `''` → `False`. `int()` converte; não arredondamos frações para torná-las válidas.


In [6]:
def quantidade_valida(texto):
    # Exige representação inteira positiva; não arredonda nem inventa valor.
    texto = texto.strip()
    return bool(re.fullmatch(r'[+-]?\d+', texto)) and int(texto) > 0


### Função `preco_valido(texto)`

**Problema:** o preço pode estar ausente, não ser numérico ou ser negativo.

**Recebe:** o texto do preço. **Retorna:** `True` ou `False`.

- `Decimal(texto.strip())` tenta interpretar o número decimal.
- `.is_finite()` rejeita infinito e `NaN`.
- `valor >= 0` permite zero e rejeita negativos.
- `try` tenta a conversão; `except InvalidOperation` trata os textos que não podem ser convertidos.

**Por que tratar a exceção?** Uma linha inválida não deve interromper a análise das demais.

**Exemplos:** `'35.50'` e `'0'` → `True`; `''` e `'-2'` → `False`.


In [7]:
def preco_valido(texto):
    # Decimal evita imprecisão binária e permite rejeitar NaN/infinito.
    try:
        valor = Decimal(texto.strip())
        return valor.is_finite() and valor >= 0
    except InvalidOperation:
        return False


### Função `converter_data(texto)`

**Problema:** uma data inválida não pode entrar na contagem por período.

**Recebe:** texto no formato `AAAA-MM-DD HH:MM:SS`. **Retorna:** um objeto `datetime` quando válido ou `None` quando a conversão falha.

`datetime.strptime(texto, formato)` interpreta o texto conforme o formato informado. `try/except ValueError` permite continuar quando houver data impossível ou texto vazio.

**Exemplos:** `'2025-10-12 14:35:00'` → objeto de data e hora; `'2025-02-30 14:35:00'` → `None`.

**Por que `None`?** Representa ausência de conversão sem inventar uma data. O texto original fica preservado em `data_venda`.


In [8]:
def converter_data(texto):
    """Datas inválidas permanecem sem conversão, sem interromper o programa."""
    try:
        return datetime.strptime(texto.strip(), '%Y-%m-%d %H:%M:%S')
    except ValueError:
        return None

### Aplicar regras de texto na tabela

| Recurso | O que faz | Por que usamos |
|---|---|---|
| `.apply(limpar_categoria)` | Chama nossa função para cada valor | Reutilizar a mesma regra em todas as categorias |
| `.ne(...)` | Compara valores e marca os diferentes | Contar mudanças em relação ao texto original |
| `~mascara` | Inverte verdadeiro e falso | Selecionar categorias que não eram vazias |
| `mascara_a & mascara_b` | Exige as duas condições | Contar apenas categorias preenchidas que mudaram |
| `.get(chave)` em um dicionário | Busca um valor; retorna `None` se não existir | Padronizar canais e evidenciar os desconhecidos |
| `lambda texto: ...` | Cria uma função curta sem nome | Fazer uma transformação simples dentro do `apply` |
| `.drop_duplicates()` | Remove linhas repetidas da seleção exibida | Mostrar os diferentes exemplos de antes e depois |
| `.reset_index(drop=True)` | Recomeça o índice sem guardar o antigo | Deixar a tabela de exemplos mais legível |

**Exemplo:** `' web '.strip().lower()` vira `'web'`; o dicionário traduz essa chave para `'Web'`.


In [9]:
categoria_vazia = original['categoria'].str.strip().eq('')
dados['categoria_original'] = original['categoria']
dados['categoria'] = original['categoria'].apply(limpar_categoria)
categoria_alterada = ~categoria_vazia & original['categoria'].ne(dados['categoria'])

mapa_canais = {'mercado': 'Mercado', 'loja física': 'Loja física', 'web': 'Web'}
dados['canal_padronizado'] = original['canal_venda'].apply(
    lambda texto: mapa_canais.get(texto.strip().lower())
)

print('Categorias vazias preenchidas:', int(categoria_vazia.sum()))
print('Categorias não vazias alteradas:', int(categoria_alterada.sum()))
display(dados[['categoria_original', 'categoria']].drop_duplicates().reset_index(drop=True))
display(dados[['canal_venda', 'canal_padronizado']].drop_duplicates().reset_index(drop=True))

Categorias vazias preenchidas: 18
Categorias não vazias alteradas: 1482


,categoria_original,categoria
0,Fantasia,fantasia
1,FANTASIA!!!,fantasia
2,Acessorio,acessorio
3,Fantasia,fantasia
4,ACESSORIO!!!,acessorio
5,Acessorio,acessorio
6,,sem categoria


,canal_venda,canal_padronizado
0,Loja física,Loja física
1,Web,Web
2,Mercado,Mercado
3,loja física,Loja física
4,mercado,Mercado
5,web,Web


## 03 · Validar sem inventar valores

- **Quantidade:** representação de inteiro maior que zero. Não arredondamos frações.
- **Preço:** número finito, preenchido e não negativo. Zero é válido.
- **Registros com problemas:** permanecem na base com indicadores; o texto original é preservado.

Usamos `Decimal` para validar preços e rejeitar valores como infinito. O operador `|` reúne duas condições: um pedido com ambos os problemas conta **uma única vez**.

### Validação numérica e conversão

| Recurso | O que faz | Por que usamos |
|---|---|---|
| `~serie_booleana` | Inverte os resultados | Transformar “válido” em indicador de “problema” |
| `mascara_a | mascara_b` | Aceita uma ou ambas as condições | Contar cada pedido com problema uma única vez |
| `int(valor)` | Converte texto inteiro para número | Usar quantidades aprovadas como números |
| `pd.array(..., dtype='Int64')` | Cria inteiros que aceitam ausências | Evitar preencher quantidades inválidas com zero |
| `Decimal(valor)` | Converte texto decimal aprovado | Manter uma representação decimal para o preço |
| `pd.Series({...}).to_frame()` | Constrói uma coluna e a apresenta como tabela | Organizar o resumo das validações |
| `.loc[mascara, colunas]` | Seleciona linhas por condição e colunas por nome | Inspecionar exatamente os registros problemáticos |

`valor_se_verdadeiro if condicao else valor_se_falso` é uma expressão condicional. Aqui só convertemos o valor se ele for válido; caso contrário, usamos ausência.


In [10]:
dados['problema_quantidade'] = ~original['quantidade'].apply(quantidade_valida)
dados['problema_preco'] = ~original['preco_unitario'].apply(preco_valido)
problema_numerico = dados['problema_quantidade'] | dados['problema_preco']

# Colunas convertidas são separadas das originais.
dados['quantidade_numero'] = pd.array([
    int(valor) if quantidade_valida(valor) else None
    for valor in original['quantidade']
], dtype='Int64')
dados['preco_numero'] = original['preco_unitario'].apply(
    lambda valor: Decimal(valor) if preco_valido(valor) else None
)

display(pd.Series({
    'Quantidade inválida': int(dados['problema_quantidade'].sum()),
    'Preço inválido': int(dados['problema_preco'].sum()),
    'Pedidos com algum problema numérico': int(problema_numerico.sum())
}, name='Pedidos').to_frame())
display(dados.loc[problema_numerico, [
    'pedido_id', 'quantidade', 'preco_unitario', 'problema_quantidade', 'problema_preco'
]])

,Pedidos
Quantidade inválida,7
Preço inválido,12
Pedidos com algum problema numérico,19


,pedido_id,quantidade,preco_unitario,problema_quantidade,problema_preco
23,1500,3,,False,True
164,630,-1,22,True,False
199,625,2,,False,True
216,1000,1,,False,True
239,1250,1,,False,True
293,500,1,,False,True
503,375,3,,False,True
571,250,4,,False,True
583,125,1,,False,True
672,840,-1,23.32,True,False


## 04 · Datas: texto, conversão e apresentação

`datetime.strptime()` interpreta a data no formato `%Y-%m-%d %H:%M:%S`. Se a data não puder ser convertida, a função retorna `None` e o processamento continua.

`strftime()` faz o caminho da apresentação: por exemplo, `12/10/2025`. Os horários são locais de São Paulo, sem conversão de fuso. **Não usamos a data de hoje como substituição.**

### Transformar datas sem interromper a análise

| Recurso | O que faz | Por que usamos |
|---|---|---|
| `[funcao(valor) for valor in coluna]` | Constrói uma lista aplicando uma função | Converter cada data e manter a ordem das linhas |
| `valor is None` | Verifica se o retorno é ausência | Marcar conversões que falharam |
| `.items()` | Percorre pares de chave e valor de um dicionário | Associar cada coluna ao formato desejado |
| `for coluna, formato in ...` | Repete um bloco | Gerar componentes das datas sem duplicar o código |
| `.strftime(formato)` | Converte uma data em texto formatado | Exibir datas brasileiras e criar ano-mês |

**Formatos:** `%Y` = ano; `%m` = mês; `%d` = dia; `%H` = hora de 00 a 23; `%M` = minuto; `%S` = segundo. Atenção: `%m` e `%M` são diferentes.

Nesta solução, os componentes extraídos com `strftime` são **textos**. Isso é suficiente para as contagens por ano-mês.


In [11]:
datas = [converter_data(texto) for texto in original['data_venda']]
dados['problema_data'] = [data is None for data in datas]

# Um laço evita repetir o mesmo código para cada componente.
formatos = {
    'ano': '%Y', 'mes': '%m', 'dia': '%d', 'hora': '%H',
    'ano_mes': '%Y-%m', 'data_br': '%d/%m/%Y'
}
for coluna, formato in formatos.items():
    dados[coluna] = [data.strftime(formato) if data else '' for data in datas]

print('Datas inválidas ou vazias:', int(dados['problema_data'].sum()))
display(dados[['pedido_id', 'data_venda', 'ano', 'mes', 'dia', 'hora', 'data_br']].head())
display(dados.loc[dados['problema_data'], ['pedido_id', 'data_venda', 'problema_data']])

Datas inválidas ou vazias: 4


,pedido_id,data_venda,ano,mes,dia,hora,data_br
0,844,2025-11-21 18:02:35,2025,11,21,18,21/11/2025
1,378,2024-10-29 18:41:26,2024,10,29,18,29/10/2024
2,953,2025-09-05 21:16:09,2025,09,05,21,05/09/2025
3,11,2024-11-08 11:29:44,2024,11,08,11,08/11/2024
4,1335,2025-08-23 11:22:05,2025,08,23,11,23/08/2025


,pedido_id,data_venda,problema_data
152,740,data_invalida,True
255,370,data_invalida,True
714,1480,data_invalida,True
884,1110,data_invalida,True


## 05 · Testar uma hipótese de negócio

> Todo pedido sem data de entrega está cancelado?

Para refutar uma afirmação sobre **todos** os registros, basta um contraexemplo. Vamos separar os pedidos sem entrega entre cancelados e outros status. Não alteraremos status nem preencheremos datas de entrega.

### Hipótese: combinar condições e procurar contraexemplos

| Recurso | O que faz | Por que usamos |
|---|---|---|
| `.str.lower()` | Converte cada texto para minúsculas | Comparar status de forma consistente |
| `.eq('cancelado')` | Testa igualdade em cada linha | Identificar cancelamentos |
| `&` e `~` | Combinam e invertem máscaras | Separar sem entrega e cancelados dos demais |
| `.any()` | Retorna verdadeiro se houver ao menos um `True` | Detectar se existe algum contraexemplo |
| `if ... else` | Escolhe qual bloco executar | Dar a conclusão adequada aos dados |
| `.head(3)` | Limita a exibição a três exemplos | Atender ao pedido do enunciado |

**Por que não usar `and` entre colunas?** No Pandas, usamos `&` para comparar condições linha a linha. Ao escrever comparações diretamente, envolva cada uma em parênteses.


In [12]:
sem_entrega = original['data_entrega'].str.strip().eq('')
cancelado = original['status_pedido'].str.strip().str.lower().eq('cancelado')
sem_entrega_cancelado = sem_entrega & cancelado
sem_entrega_outros = sem_entrega & ~cancelado

print('Total de cancelados:', int(cancelado.sum()))
print('Sem entrega e cancelados:', int(sem_entrega_cancelado.sum()))
print('Sem entrega e outros status:', int(sem_entrega_outros.sum()))

if sem_entrega_outros.any():
    print('Hipótese FALSA nesta base: existem contraexemplos.')
else:
    print('Hipótese confirmada nesta base.')

display(original.loc[sem_entrega_outros,
    ['pedido_id', 'status_pedido', 'data_entrega']].head(3))

Total de cancelados: 109
Sem entrega e cancelados: 109
Sem entrega e outros status: 34
Hipótese FALSA nesta base: existem contraexemplos.


,pedido_id,status_pedido,data_entrega
49,1170,Concluído,
117,990,Concluído,
152,740,Concluído,


## 06 · Panorama da qualidade

As máscaras booleanas marcam quais linhas satisfazem cada condição. A soma de `True` conta ocorrências. A união das máscaras conta pedidos afetados sem duplicá-los.

### Resumir com um dicionário

O dicionário `indicadores` associa **nome da métrica → valor**. `pd.Series(indicadores, name='Total')` transforma esses pares em uma coluna; `.to_frame()` apresenta o resultado como tabela.

`int(mascara.sum())` entrega uma contagem inteira. A expressão `problema_numerico | dados['problema_data']` representa a união dos problemas: uma linha que aparece em dois grupos continua contando uma única vez.


In [13]:
indicadores = {
    'Linhas lidas': len(dados),
    'Categorias vazias preenchidas': int(categoria_vazia.sum()),
    'Categorias não vazias alteradas': int(categoria_alterada.sum()),
    'Quantidades inválidas': int(dados['problema_quantidade'].sum()),
    'Preços inválidos': int(dados['problema_preco'].sum()),
    'Pedidos com problemas numéricos': int(problema_numerico.sum()),
    'Datas inválidas ou vazias': int(dados['problema_data'].sum()),
    'Pedidos com problema numérico ou de data': int((problema_numerico | dados['problema_data']).sum()),
    'Cancelados': int(cancelado.sum()),
    'Sem entrega e cancelados': int(sem_entrega_cancelado.sum()),
    'Sem entrega e outros status': int(sem_entrega_outros.sum())
}
display(pd.Series(indicadores, name='Total').to_frame())

,Total
Linhas lidas,1500
Categorias vazias preenchidas,18
Categorias não vazias alteradas,1482
Quantidades inválidas,7
Preços inválidos,12
Pedidos com problemas numéricos,19
Datas inválidas ou vazias,4
Pedidos com problema numérico ou de data,23
Cancelados,109
Sem entrega e cancelados,109


## 07 · Descoberta por canal

Contamos **pedidos registrados, com todos os status**. Para esta contagem, o campo que precisa estar válido é o canal; um problema de preço não elimina o pedido desta análise.

### Contagens e percentuais por canal

| Recurso | O que faz | Por que usamos |
|---|---|---|
| `.value_counts()` | Conta ocorrências de cada valor; omite nulos por padrão | Contar pedidos por canal reconhecido |
| `.rename_axis('Canal')` | Dá um nome ao índice | Identificar o significado das linhas |
| `.to_frame('Pedidos')` | Transforma a Series em DataFrame | Criar uma tabela com nome de coluna claro |
| `contagem / total * 100` | Calcula participação percentual | Comparar o peso de cada canal |
| `.round(1)` | Arredonda para uma casa decimal | Facilitar a leitura dos percentuais |
| `.isna().sum()` | Conta canais sem mapeamento | Informar exclusões explicitamente |

O denominador é o total de pedidos com canal reconhecido. Nesta base, todos os canais foram reconhecidos.


In [14]:
por_canal = dados['canal_padronizado'].value_counts().rename_axis('Canal').to_frame('Pedidos')
por_canal['Participação (%)'] = (por_canal['Pedidos'] / por_canal['Pedidos'].sum() * 100).round(1)
display(por_canal)
print('Excluídos por canal não reconhecido:', int(dados['canal_padronizado'].isna().sum()))

,Pedidos,Participação (%)
Canal,,
Web,666,44.4
Loja física,551,36.7
Mercado,283,18.9


Excluídos por canal não reconhecido: 0


### 💡 Interpretação

A **Web registra 666 pedidos (44,4%)**, seguida da Loja física, com 551, e do Mercado, com 283. O canal digital concentra a maior parcela dos registros e merece atenção no planejamento operacional.

**Limitação:** a contagem inclui cancelamentos e devoluções. Não permite concluir qual canal gera maior receita, mais unidades ou mais vendas concluídas. Nenhum pedido foi excluído por canal.

## 08 · Descoberta por período

Usamos **ano e mês juntos**, evitando misturar outubro de 2024 com outubro de 2025. Excluímos apenas registros sem data de venda válida e mantemos todos os status.

### Contagens por ano e mês

| Recurso | O que faz | Por que usamos |
|---|---|---|
| `.loc[~dados['problema_data'], 'ano_mes']` | Seleciona ano-mês apenas das datas válidas | Excluir períodos desconhecidos |
| `.value_counts()` | Conta registros de cada período | Medir pedidos registrados |
| `.sort_index()` | Ordena os rótulos do índice | Mostrar `AAAA-MM` em ordem cronológica |
| `:.1%` em uma f-string | Exibe uma fração como percentual | Apresentar a participação com uma casa decimal |

**Por que ano-mês?** Agrupar somente por `'10'` somaria os outubros de anos diferentes. A comparação entre outubros usa `(410 / 393 - 1) × 100` e descreve os resultados desta base específica.


In [15]:
por_periodo = (dados.loc[~dados['problema_data'], 'ano_mes']
               .value_counts().sort_index().rename_axis('Ano-mês').to_frame('Pedidos'))
display(por_periodo)
print('Excluídos por data inválida:', int(dados['problema_data'].sum()))

outubro = dados['mes'].eq('10').sum()
print(f'Outubros: {outubro} de {por_periodo["Pedidos"].sum()} pedidos datáveis '
      f'({outubro / por_periodo["Pedidos"].sum():.1%}).')
print(f'Variação entre outubros: {(410 / 393 - 1):.1%}.')

,Pedidos
Ano-mês,
2024-08,77
2024-09,104
2024-10,393
2024-11,174
2025-08,62
2025-09,94
2025-10,410
2025-11,182


Excluídos por data inválida: 4
Outubros: 803 de 1496 pedidos datáveis (53.7%).
Variação entre outubros: 4.3%.


### 💡 Interpretação

**Outubro lidera nos dois anos:** 393 pedidos em 2024 e 410 em 2025. A diferença é de 17 registros, ou 4,3%. Os dois outubros somam **803 pedidos, 53,7% dos 1.496 com data válida**.

A concentração é compatível com a proximidade do Halloween e pode orientar o planejamento de estoque.

**Limitação:** quatro datas foram excluídas e todos os status foram incluídos. Dois anos de dados fictícios não comprovam sazonalidade no mercado real nem causalidade.

## 09 · Conferir e exportar

Vamos conferir se as contagens reconciliam com o total e salvar uma **nova cópia analisada**. O arquivo de entrada permanece intacto.

### Conferências e arquivos de saída

| Recurso | O que faz | Por que usamos |
|---|---|---|
| `assert condicao, mensagem` | Interrompe se uma conferência falhar | Evidenciar divergências nas contagens ou nos dados |
| `.equals(outro_dataframe)` | Compara tabelas | Confirmar que a tabela original coincide com a releitura |
| `.mkdir(exist_ok=True)` | Cria a pasta se necessário | Organizar as saídas sem erro caso a pasta já exista |
| `.to_csv()` | Grava uma tabela em CSV | Guardar resultados em arquivos separados da entrada |
| `index=False` | Não grava o índice do DataFrame | Evitar uma coluna técnica extra no arquivo analisado |
| `.resolve()` | Obtém o caminho absoluto | Mostrar onde encontrar as saídas |

Nas tabelas por canal e período, o índice é mantido porque contém os nomes dos grupos.


In [16]:
assert original.equals(pd.read_csv(
    ARQUIVO, sep=',', encoding='utf-8', dtype=str, keep_default_na=False
)), 'O arquivo original não deve mudar.'
assert por_canal['Pedidos'].sum() + dados['canal_padronizado'].isna().sum() == len(dados)
assert por_periodo['Pedidos'].sum() + dados['problema_data'].sum() == len(dados)
print('Conferências concluídas: contagens reconciliadas e entrada preservada.')

SAIDA = Path('resultados_notebook')
SAIDA.mkdir(exist_ok=True)
dados.to_csv(SAIDA / 'dados_analisados.csv', index=False, encoding='utf-8')
por_canal.to_csv(SAIDA / 'pedidos_por_canal.csv', encoding='utf-8')
por_periodo.to_csv(SAIDA / 'pedidos_por_periodo.csv', encoding='utf-8')
print('Resultados gravados em:', SAIDA.resolve())

Conferências concluídas: contagens reconciliadas e entrada preservada.
Resultados gravados em: /workspace/scratch/16ded07d4df7/resolucao_halloween/resultados_notebook


## 10 · Relatório e aprendizados

O relatório abaixo contém evidências, interpretações e limitações. A contagem utiliza `len(texto)`, incluindo espaços e quebras de linha; o texto foi conferido nesta resolução assistida por IA.

**Qualidade e Machine Learning:** limpar os dados melhora a consistência, mas não garante representatividade nem elimina overfitting ou underfitting. A escolha do modelo e a avaliação também importam. Este projeto não exige treinamento de modelos.

### Relatório: texto, tamanho e apresentação

| Recurso | O que faz | Por que usamos |
|---|---|---|
| `len(relatorio)` | Conta caracteres da string | Conferir o limite de 1.500 a 3.000, com espaços e quebras |
| `1500 <= tamanho <= 3000` | Testa o intervalo permitido | Verificar o requisito do enunciado |
| `.write_text(..., encoding='utf-8')` | Grava um texto em arquivo | Criar `report.md` com o conteúdo final |
| `Markdown(relatorio)` | Prepara o texto Markdown para exibição | Apresentar o relatório formatado no notebook |

Se você editar o relatório, execute novamente esta célula para atualizar a contagem e o arquivo.


In [17]:
relatorio = '# Relatório — Festa em Cena\n\nA análise do arquivo identificou 1.500 linhas e 1.500 pedidos distintos. Cada linha representa um pedido de um produto, que pode incluir várias unidades. A base é fictícia e cobre agosto a novembro de 2024 e 2025. As contagens incluem Concluído, Cancelado e Devolvido e representam pedidos registrados, não receita nem vendas concluídas.\n\nNa limpeza, 18 categorias vazias foram preenchidas com "sem categoria". As outras 1.482 categorias foram alteradas pela padronização para minúsculas, remoção de espaços nas extremidades e, quando necessário, retirada de pontuação com expressão regular. Esse total inclui mudanças apenas de maiúsculas para minúsculas; não significa que todas essas categorias tinham pontuação indevida.\n\nForam encontrados 7 pedidos com quantidade inválida e 12 com preço inválido. A união desses grupos contém 19 pedidos; nesta base não há sobreposição. Outros 4 pedidos apresentam data de venda inválida ou vazia, totalizando 23 pedidos com pelo menos um problema numérico ou temporal. Os valores originais foram preservados e os problemas sinalizados, sem imputação de números ou datas.\n\nA hipótese de que todo pedido sem data de entrega é cancelado foi refutada: 109 pedidos sem entrega estão cancelados, mas 34 possuem outros status. Os pedidos 1170, 990 e 740, por exemplo, estão como Concluído e não têm data de entrega. A ausência desse campo não comprova cancelamento nem permite afirmar que a entrega não ocorreu; recomenda-se investigar o preenchimento antes de usar esse campo como indicador operacional.\n\nPor canal, a Web registra 666 pedidos (44,4%), a Loja física 551 (36,7%) e o Mercado 283 (18,9%). Todos os canais foram reconhecidos, sem exclusões. A Web concentra a maior parcela dos registros e merece atenção no planejamento operacional. Entretanto, sem avaliar valores, unidades e status separadamente, não se pode concluir que ela gere maior faturamento ou mais vendas efetivas.\n\nPor período, outubro lidera nos dois anos: 393 pedidos em 2024 e 410 em 2025, aumento de 17 registros (4,3%). Somados, os dois outubros representam 803 dos 1.496 pedidos com data válida (53,7%). O padrão é compatível com a concentração de procura próxima ao Halloween e pode orientar o planejamento de estoque. A análise exclui 4 datas inválidas e inclui todos os status. Dois anos de dados fictícios não permitem comprovar sazonalidade no mercado real nem atribuir causalidade ao evento.\n\nA qualidade dos dados é uma etapa anterior à modelagem. A limpeza melhora a consistência, mas não garante representatividade ou elimina overfitting e underfitting, que também dependem do modelo e de sua avaliação.\n'

print('Caracteres, incluindo espaços e quebras de linha:', len(relatorio))
assert 1500 <= len(relatorio) <= 3000
Path('report.md').write_text(relatorio, encoding='utf-8')
display(Markdown(relatorio))

Caracteres, incluindo espaços e quebras de linha: 2657


# Relatório — Festa em Cena

A análise do arquivo identificou 1.500 linhas e 1.500 pedidos distintos. Cada linha representa um pedido de um produto, que pode incluir várias unidades. A base é fictícia e cobre agosto a novembro de 2024 e 2025. As contagens incluem Concluído, Cancelado e Devolvido e representam pedidos registrados, não receita nem vendas concluídas.

Na limpeza, 18 categorias vazias foram preenchidas com "sem categoria". As outras 1.482 categorias foram alteradas pela padronização para minúsculas, remoção de espaços nas extremidades e, quando necessário, retirada de pontuação com expressão regular. Esse total inclui mudanças apenas de maiúsculas para minúsculas; não significa que todas essas categorias tinham pontuação indevida.

Foram encontrados 7 pedidos com quantidade inválida e 12 com preço inválido. A união desses grupos contém 19 pedidos; nesta base não há sobreposição. Outros 4 pedidos apresentam data de venda inválida ou vazia, totalizando 23 pedidos com pelo menos um problema numérico ou temporal. Os valores originais foram preservados e os problemas sinalizados, sem imputação de números ou datas.

A hipótese de que todo pedido sem data de entrega é cancelado foi refutada: 109 pedidos sem entrega estão cancelados, mas 34 possuem outros status. Os pedidos 1170, 990 e 740, por exemplo, estão como Concluído e não têm data de entrega. A ausência desse campo não comprova cancelamento nem permite afirmar que a entrega não ocorreu; recomenda-se investigar o preenchimento antes de usar esse campo como indicador operacional.

Por canal, a Web registra 666 pedidos (44,4%), a Loja física 551 (36,7%) e o Mercado 283 (18,9%). Todos os canais foram reconhecidos, sem exclusões. A Web concentra a maior parcela dos registros e merece atenção no planejamento operacional. Entretanto, sem avaliar valores, unidades e status separadamente, não se pode concluir que ela gere maior faturamento ou mais vendas efetivas.

Por período, outubro lidera nos dois anos: 393 pedidos em 2024 e 410 em 2025, aumento de 17 registros (4,3%). Somados, os dois outubros representam 803 dos 1.496 pedidos com data válida (53,7%). O padrão é compatível com a concentração de procura próxima ao Halloween e pode orientar o planejamento de estoque. A análise exclui 4 datas inválidas e inclui todos os status. Dois anos de dados fictícios não permitem comprovar sazonalidade no mercado real nem atribuir causalidade ao evento.

A qualidade dos dados é uma etapa anterior à modelagem. A limpeza melhora a consistência, mas não garante representatividade ou elimina overfitting e underfitting, que também dependem do modelo e de sua avaliação.


---
### ✅ Checklist de compreensão

- Consigo explicar por que lemos as colunas como texto?
- Entendo por que 1.482 categorias alteradas não significam 1.482 erros de pontuação?
- Sei diferenciar ocorrências por campo de pedidos afetados?
- Consigo mostrar um contraexemplo à hipótese das entregas?
- Entendo as exclusões e limitações das duas descobertas?

**Para o vídeo de 5 a 7 minutos:** apresente o problema, mostre as funções, execute as células principais e explique as descobertas com suas palavras. Relate apenas branches, dificuldades e decisões que realmente fizeram parte do seu desenvolvimento.